<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex07.2-heat-and-wave/Ex07.2_04_missing_condition.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_07.2 · Notebook 04 — What Happens If You Forget One

**Paired with L7.2 · Fundamental PDEs**

This is the most important notebook in Ex_07.

You are going to solve the panel again, correctly in every respect except one:
the velocity condition is left out. The PDE is right. Both boundary conditions
are right. The displacement condition is right. Only $u_t(x,y,0)$ is missing.

The loss will converge beautifully. The answer will be nonsense. And nothing
in the training output will tell you.

## Why this works so cleanly here

The panel starts **flat**. So $u \equiv 0$ everywhere, for all time:

* satisfies $u_{tt} = c^2\nabla^2 u$ exactly — both sides are zero;
* satisfies $u = 0$ on all four edges exactly;
* satisfies $u(x,y,0) = 0$ exactly.

Every term you kept is satisfied by the trivial solution. The only thing ruling
it out is the term you dropped.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex07.2-heat-and-wave/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

## 1 · The under-determined problem

### Your turn

In [ ]:
# TODO: build the loss WITHOUT the velocity term, and train.
#
#   Same residual as notebook 03. Same boundary term. Same displacement term.
#   Simply omit the iv term:
#
#   def make_loss_missing(model, xyt_f, xyt_b, xyt_0, u0, w_b=1.0, w_u=1.0):
#       def loss():
#           f = wave_residual(model, xyt_f) / R_SCALE
#           b = model(xyt_b) / U_SCALE
#           iu = (model(xyt_0) - u0) / U_SCALE
#           return mse(f) + w_b*mse(b) + w_u*mse(iu)
#       return loss
#
#   set_seed(88); model_missing = MLP(n_in=3, n_hidden=48, n_layers=5)
#   Same points as notebook 03, same schedule.
#   Record history_missing.

raise NotImplementedError("Train the panel with the velocity condition omitted")

In [ ]:
plot_curves(history_missing, title="the panel, velocity condition omitted")
plt.show()

print(f"final loss, this notebook : {history_missing['lbfgs'][-1]:.3e}")
nb03 = np.load(os.path.join("Ex07.2_outputs", "nb03_panel.npz"))
print(f"final loss, notebook 03   : {nb03['lbfgs'][-1]:.3e}")

**What you should see.** A loss that falls further and faster than notebook
03's — often by several orders of magnitude.

Read that again. **The wrong problem converged better than the right one.**
Of course it did: it is an easier problem, and the function it is converging
to is the simplest one imaginable.

---

## 2 · Look at what it produced

### Your turn

In [ ]:
# TODO: evaluate the model and quantify how much it moves.
#
#   ts = np.linspace(0.0, pb.WAVE_T_END, 400)
#   mid = pb.L_PANEL / 2
#   q = np.stack([np.full_like(ts, mid), np.full_like(ts, mid), ts], axis=1)
#   with torch.no_grad():
#       centre_missing = to_numpy(model_missing(to_tensor(q))).ravel()
#
#   Record:
#       peak_missing = np.abs(centre_missing).max()
#       peak_exact   = np.abs(pb.wave_exact(q[:,0], q[:,1], ts)).max()
#       fraction     = peak_missing / peak_exact

raise NotImplementedError("Evaluate the under-determined model")

In [ ]:
pb.plot_time_history(
    ts,
    {"velocity condition omitted": centre_missing * 1e3,
     "both conditions (nb 03)": nb03["centre_pred"] * 1e3,
     "exact": nb03["centre_exact"] * 1e3},
    title="The panel that was struck, and the panel that was not",
    ylabel="centre displacement [mm]")
plt.show()

print(f"  peak deflection, exact           : {peak_exact*1e3:.4f} mm")
print(f"  peak deflection, model           : {peak_missing*1e3:.6f} mm")
print(f"  fraction of the real motion      : {fraction*100:.3f} %")
print()
print(f"  and the loss said                : {history_missing['lbfgs'][-1]:.3e}")

**What you should see.** A flat line at zero, against a sinusoid of amplitude
0.56 mm. The model captures well under a percent of the real motion, and often
far less.

**The panel was struck at half a metre per second and the model says it never
moved.**

---

## 3 · Every check you might have run, and what it says

### Your turn

In [ ]:
# TODO: run the diagnostics a careful person would run, and record each.
#
#   On the under-determined model, compute:
#
#     pde_res   : RMS of wave_residual on fresh interior points, / R_SCALE
#     edge_err  : max |u| on fresh boundary-in-time points, in metres
#     ic_disp   : max |u - 0| on a fresh initial slice, in metres
#     ic_vel    : max |u_t - v0| on that slice, in m/s     <- the omitted one
#
#   Put them in diag = {name: value}.
#
# Use fresh points, not the training set. Passing on your own collocation
# points is not evidence of anything.

raise NotImplementedError("Run the diagnostics on the under-determined model")

In [ ]:
print(error_table(
    [["PDE residual (RMS, scaled)", f"{diag['pde_res']:.3e}", "looks excellent"],
     ["edges, max |u|", f"{diag['edge_err']:.3e} m", "looks excellent"],
     ["u(x,y,0), max error", f"{diag['ic_disp']:.3e} m", "looks excellent"],
     ["u_t(x,y,0), max error", f"{diag['ic_vel']:.4f} m/s",
      "THE ONLY ONE THAT KNOWS"]],
    ["check", "value", "verdict"]))

**What you should see.** Three checks that pass magnificently, and one that
fails completely — the one you did not put in the loss.

This is the lesson of L7.2 and it generalises well beyond wave equations:

> **A converged residual tells you the network solves the problem you posed.
> It says nothing about whether you posed the right problem.**

In Ex_07.1 you could always compare against an exact solution. In L8 onward you
often cannot, and the residual is the only number you have. It is not enough.
Counting conditions — one per order in time, per independent variable — is a
five-second check that would have caught this.

---

## 4 · Two more ways to be under-determined

The missing initial condition is the cleanest example. It is not the only one.

### Your turn

In [ ]:
# TODO: pick ONE of these and demonstrate it. A paragraph and a figure.
#
#   (a) Omit ONE of the four edges from the boundary term. The panel now has a
#       free edge it was never told about. Does the model notice? Does the
#       residual?
#
#   (b) Keep both initial conditions but sample the initial slice with only 20
#       points instead of 800. The condition is present but weakly enforced.
#       Where does the error appear -- and is it obvious from the loss?
#
#   Record your figure and a short verdict string.

raise NotImplementedError("Demonstrate one further way to be under-determined")

## 5 · Save

In [ ]:
path = os.path.join("Ex07.2_outputs", "nb04_missing_ic.npz")
np.savez(path,
         centre_missing=centre_missing, ts=ts,
         peak_missing=peak_missing, peak_exact=peak_exact, fraction=fraction,
         pde_res=diag["pde_res"], edge_err=diag["edge_err"],
         ic_disp=diag["ic_disp"], ic_vel=diag["ic_vel"],
         final_loss=history_missing["lbfgs"][-1],
         adam=history_missing["adam"], lbfgs=history_missing["lbfgs"])
print("wrote", path)

## 6 · Before you move on

1. State, in one sentence, why $u \equiv 0$ satisfied everything you kept.
2. The under-determined problem reached a **lower** loss than the correct one.
   Explain why that is not surprising, and why it is dangerous.
3. Give the counting rule for how many initial conditions a PDE needs, and
   apply it to the die and to the panel.
4. In L10 you will solve a battery model with no exact solution available. List
   two checks you could still run that would catch a mis-posed problem.

Next: **notebook 05**, the report.